<img src="./static/imo_health.png" alt="IMO Health Logo" width="300"/>

---

# IMO Health MCP Gateway — LangChain Agent

This notebook creates a LangChain ReAct agent backed by **AWS Bedrock (Claude)** that connects to the **IMO Health MCP Gateway** and interacts with its tools.

MCP Server: `https://api.imohealth.com/mcp`

## Step 1: Install Packages and Load Configuration

Reads `mcp.client_id` from `config.json` and fetches `mcp.client_secret` from AWS SSM Parameter Store (`/imo/mcp/client_secret`). On SageMaker the execution role is used automatically — no extra credentials needed.

In [ ]:
%pip install -q --upgrade \
    "langchain-core>=1.0.0" \
    "langchain>=1.0.0" \
    "langchain-openai>=0.4.0" \
    "langchain-aws>=0.2.0" \
    "langchain-mcp-adapters>=0.1.5" \
    "langgraph>=0.2.0" \
    "mcp>=1.0.0" \
    boto3 requests nest_asyncio

# Restart the kernel after running this cell.

In [ ]:
import json
import pathlib
import asyncio
import nest_asyncio
import requests

nest_asyncio.apply()

# --- Load config.json ---
candidates = [
    pathlib.Path('config.json'),
    pathlib.Path.cwd() / 'config.json',
    pathlib.Path.cwd().parent / 'config.json'
]
cfg_path = next((p for p in candidates if p.exists()), None)
if cfg_path is None:
    raise FileNotFoundError('config.json not found. Copy config.json.template to config.json.')

with open(cfg_path, 'r', encoding='utf-8') as f:
    cfg = json.load(f)

mcp_cfg = cfg.get('mcp', {})

MCP_CLIENT_ID     = mcp_cfg.get('client_id')  or '2DekQGY8Q5rM19bvsDk8d7Oh2MdXMcP6'
MCP_CLIENT_SECRET = mcp_cfg.get('client_secret') or ''
TOKEN_URL         = mcp_cfg.get('token_url')  or 'https://api.imohealth.com/oauth/token'
MCP_SERVER_URL    = mcp_cfg.get('server_url') or 'https://api.imohealth.com/mcp'
BEDROCK_MODEL     = mcp_cfg.get('bedrock_model_id') or 'c3ccht7cnzsf'
BEDROCK_REGION    = mcp_cfg.get('aws_region') or 'us-east-1'

if not MCP_CLIENT_SECRET:
    raise ValueError('client_secret is missing in config.json under the "mcp" section.')

print(f'Config loaded from : {cfg_path.resolve()}')
print(f'Bedrock Model      : {BEDROCK_MODEL}')
print(f'AWS Region         : {BEDROCK_REGION}')
print(f'Client ID          : {MCP_CLIENT_ID}')
print(f'Token URL          : {TOKEN_URL}')
print(f'MCP Server URL     : {MCP_SERVER_URL}')

## Step 2: Obtain OAuth 2.0 Access Token

Uses `client_credentials` grant against `https://api.imohealth.com/mcp/token` with the configured scopes.

In [ ]:
def get_token(client_id: str, client_secret: str) -> str:
    payload = {
        'grant_type':    'client_credentials',
        'client_id':     client_id,
        'client_secret': client_secret,
        'audience':      'https://api.imohealth.com'
    }
    resp = requests.post(TOKEN_URL, json=payload, timeout=30)
    resp.raise_for_status()
    return resp.json()['access_token']

access_token = get_token(MCP_CLIENT_ID, MCP_CLIENT_SECRET)
print('Token acquired (prefix):', access_token[:20] + '...')

## Step 3: Connect to MCP Gateway and Discover Tools

Connects to the IMO Health MCP server and lists all available tools.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

MCP_CLIENT_CONFIG = {
    'mcp-gateway': {
        'url':       MCP_SERVER_URL,
        'transport': 'streamable_http',
        'headers':   {'Authorization': f'Bearer {access_token}'},
    }
}

async def list_mcp_tools():
    client = MultiServerMCPClient(MCP_CLIENT_CONFIG)
    tools = await client.get_tools()
    print(f'Discovered {len(tools)} MCP tools:')
    for t in tools:
        print(f'  - {t.name}: {t.description}')

asyncio.run(list_mcp_tools())

## Step 4: Set Up AWS Bedrock LLM (Claude)

On SageMaker the execution role provides Bedrock access automatically — no extra credentials needed.

In [ ]:
from langchain_aws import ChatBedrock
from langchain.agents import create_agent
import boto3 as _boto3

def _resolve_model_id(model_id: str, region: str) -> str:
    """If model_id is a bare application inference profile ID (no dots, no arn:),
    resolve it to its full ARN so Bedrock accepts it."""
    if model_id.startswith('arn:') or '.' in model_id:
        return model_id
    sts = _boto3.client('sts', region_name=region)
    account_id = sts.get_caller_identity()['Account']
    return f'arn:aws:bedrock:{region}:{account_id}:application-inference-profile/{model_id}'

RESOLVED_MODEL = _resolve_model_id(BEDROCK_MODEL, BEDROCK_REGION)

# provider must be set explicitly when model_id is an ARN
MODEL_PROVIDER = 'anthropic' if RESOLVED_MODEL.startswith('arn:') else None

llm = ChatBedrock(
    model_id=RESOLVED_MODEL,
    region_name=BEDROCK_REGION,
    **({"provider": MODEL_PROVIDER} if MODEL_PROVIDER else {}),
)

print(f'LLM ready: {RESOLVED_MODEL} (region: {BEDROCK_REGION})')

## Step 5: Interact with the Agent

The MCP session stays open for the full duration of each agent invocation so tool calls work correctly.

The agent uses a **ReAct** loop: it reasons about which MCP tool to call, calls it, observes the result, and repeats until it has a final answer.

In [ ]:
async def ask_agent(query: str) -> str:
    """Run the LangChain agent for a single query within an active MCP session."""
    client = MultiServerMCPClient(MCP_CLIENT_CONFIG)
    tools = await client.get_tools()
    agent = create_agent(llm, tools)
    response = await agent.ainvoke({'messages': [{'role': 'user', 'content': query}]})
    return response['messages'][-1].content

In [ ]:
# --- Sample query 1: search for a clinical term ---
result = asyncio.run(ask_agent('Search for hypertension and return the top 3 results'))
print(result)

In [ ]:
# --- Sample query 2: normalize a clinical term ---
result = asyncio.run(ask_agent('Normalize the term "chest pain" and get all the treatment options for chest pain'))
print(result)

In [ ]:
# --- Interactive: enter your own query ---
user_query = input('Enter your query: ')
result = asyncio.run(ask_agent(user_query))
print(result)